In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import pandas as pd

df = pd.read_parquet("/rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_st_test_corpus_data.parquet")

# remove any rows with empty query or duplicate queries
df = df[df['query'].notna() & (df['query'] != '')]
df = df.drop_duplicates(subset=['query'])

# also remove datapoints with empty context
df = df[df['context'].notna() & (df['context'] != '')]


In [3]:
df.head()

,query,context,type,synthesized,source,metadata,url1
727493,SHEBA Ice Camp environmental monitoring,Description: NCAR portable automated mesonet (...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C1214601988-SCIOPS"", ""tr...",https://cmr.earthdata.nasa.gov/search/concepts...
489836,bio-geochemical cycles Ross Sea,Description: The data sets include measurement...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C1214593766-SCIOPS"", ""tr...",https://cmr.earthdata.nasa.gov/search/concepts...
869447,Pearl Harbor oceanographic data,Description: This dataset contains oceanograph...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C2089378855-NOAA_NCEI"", ...",https://cmr.earthdata.nasa.gov/search/concepts...
668676,optical backscatter measurement techniques,Description: Two hydrographic surveys were per...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C1214155000-SCIOPS"", ""tr...",https://cmr.earthdata.nasa.gov/search/concepts...
723711,gravity and magnetic field data integration,Description: This data set contains underway g...,search_term-document,True,CMR,"{""id"": ""/SDE/CMR_API/|C1214611760-SCIOPS"", ""tr...",https://cmr.earthdata.nasa.gov/search/concepts...


In [4]:
df.shape

(113531, 7)

In [5]:
df["type"].value_counts()

type
search_term-document    90308
question-answer         16855
title-description        6368
Name: count, dtype: int64

In [6]:
df["source"].value_counts()

source
CMR               74295
SDE_general_v2    16945
SDE_general_v3    16852
PDS                5439
Name: count, dtype: int64

In [7]:
df[["type", "source"]].value_counts().sort_index(ascending=False)

type                  source        
title-description     PDS                 974
                      CMR                5394
search_term-document  SDE_general_v3     8645
                      SDE_general_v2     8297
                      PDS                4465
                      CMR               68901
question-answer       SDE_general_v3     8207
                      SDE_general_v2     8648
Name: count, dtype: int64

In [8]:
n_title_desc = 500
n_search_doc = 500
n_qa = 500

# subsample for title desc with 1:5 ratio for PDS and CMR
n_title_desc_pds = n_title_desc // 6
n_title_desc_cmr = n_title_desc - n_title_desc_pds

print("For title desc:")
print(f"PDS: {n_title_desc_pds} \nCMR: {n_title_desc_cmr}")

# subsample for search doc with equal ratios
n_search_doc_pds = n_search_doc // 4
n_search_doc_cmr = n_search_doc // 4
n_search_doc_sde1 = n_search_doc // 4
n_search_doc_sde2 = n_search_doc - (n_search_doc_pds + n_search_doc_cmr + n_search_doc_sde1)
print("For search doc:")
print(f"PDS: {n_search_doc_pds} \nCMR: {n_search_doc_cmr} \nSDE1: {n_search_doc_sde1} \nSDE2: {n_search_doc_sde2}")

# subsample for qa with equal ratios
n_qa_sde1 = n_qa // 2
n_qa_sde2 = n_qa - n_qa_sde1
print("For QA:")
print(f"SDE1: {n_qa_sde1} \nSDE2: {n_qa_sde2}")

# taking subsample for title desc
df_title_desc_pds = df[(df["type"] == "title-description") & (df["source"] == "PDS")].sample(n=n_title_desc_pds, random_state=42)
df_title_desc_cmr = df[(df["type"] == "title-description") & (df["source"] == "CMR")].sample(n=n_title_desc_cmr, random_state=42)
df_title_desc = pd.concat([df_title_desc_pds, df_title_desc_cmr], ignore_index=False)


# taking subsample for search doc
df_search_doc_pds = df[(df["type"] == "search_term-document") & (df["source"] == "PDS")].sample(n=n_search_doc_pds, random_state=42)
df_search_doc_cmr = df[(df["type"] == "search_term-document") & (df["source"] == "CMR")].sample(n=n_search_doc_cmr, random_state=42)
df_search_doc_sde1 = df[(df["type"] == "search_term-document") & (df["source"] == "SDE_general_v2")].sample(n=n_search_doc_sde1, random_state=42)
df_search_doc_sde2 = df[(df["type"] == "search_term-document") & (df["source"] == "SDE_general_v3")].sample(n=n_search_doc_sde2, random_state=42)
df_search_doc = pd.concat([df_search_doc_pds, df_search_doc_cmr, df_search_doc_sde1, df_search_doc_sde2], ignore_index=False)

# taking subsample for qa
df_qa_sde1 = df[(df["type"] == "question-answer") & (df["source"] == "SDE_general_v2")].sample(n=n_qa_sde1, random_state=42)
df_qa_sde2 = df[(df["type"] == "question-answer") & (df["source"] == "SDE_general_v3")].sample(n=n_qa_sde2, random_state=42)
df_qa = pd.concat([df_qa_sde1, df_qa_sde2], ignore_index=False)

query_df = pd.concat([df_title_desc, df_search_doc, df_qa], ignore_index=False)

# get negative corpus: these are datapoints not in df_query
negative_corpus_df = df[~df.index.isin(query_df.index)]

# sampliong negative corpus
# negative_corpus_df = negative_corpus_df.sample(n=50_000, random_state=42)

For title desc:
PDS: 83 
CMR: 417
For search doc:
PDS: 125 
CMR: 125 
SDE1: 125 
SDE2: 125
For QA:
SDE1: 250 
SDE2: 250


In [9]:
# sperating out the meta data

# for query
# - type
# - synthesized


# for cocrpus
# - source
# - url0


In [10]:
negative_corpus_df.shape, query_df.shape

((112031, 7), (1500, 7))

# converting it to standard jsonal format

In [11]:
import pandas as pd
import json
import os

def create_jsonl_with_negative_corpus(positive_df: pd.DataFrame, negative_df: pd.DataFrame, output_dir: str):
    """
    Generates a dataset for information retrieval, creating corpus, queries, and qrels files.

    This function takes two DataFrames, one with positive query-context pairs and one with
    negative (irrelevant) contexts, and processes them into a structured dataset format.
    It creates three main files:
    1.  `corpus.jsonl`: Contains all unique contexts from both positive and negative dataframes,
        each with a unique ID.
    2.  `queries.jsonl`: Contains all unique queries from the positive dataframe, each with a
        unique ID.
    3.  `qrels/test.tsv`: A tab-separated file mapping query IDs to their relevant
        corpus IDs, based on the positive pairs.

    Args:
        positive_df (pd.DataFrame): DataFrame containing the positive examples.
            Expected columns are: 'query', 'context', 'source', 'url1', 'type', and 'synthesized'.
        negative_df (pd.DataFrame): DataFrame containing negative or irrelevant contexts to be
            added to the corpus. Expected columns include: 'context', 'source', and 'url1'.
        output_dir (str): The path to the directory where the output files will be saved.
            The directory and a 'qrels' subdirectory will be created if they do not exist.
            
    Side Effects:
        - Creates the specified `output_dir` and a `qrels` subdirectory within it.
        - Writes `corpus.jsonl`, `queries.jsonl`, and `qrels/test.tsv` to the disk.
        - Prints status messages to the console during file generation.
    """
    # --- 1. Create Output Directories ---
    qrels_dir = os.path.join(output_dir, 'qrels')
    if not os.path.exists(qrels_dir):
        os.makedirs(qrels_dir)
        print(f"Created directory: {qrels_dir}")

    # --- 2. Process Corpus ---
    # concat the positive and negative corpus dataframes
    total_corpus_df = pd.concat([positive_df, negative_df], ignore_index=False)
    # Get unique contexts to create the corpus
    corpus_df = total_corpus_df[['context', 'source', 'url1']].drop_duplicates(subset=['context']).reset_index(drop=True)
    
    # Create a mapping from context text to a unique corpus ID
    context_to_id = {row['context']: f"c{index}" for index, row in corpus_df.iterrows()}
    
    corpus_filepath = os.path.join(output_dir, 'corpus.jsonl')
    print(f"Generating {corpus_filepath}...")
    with open(corpus_filepath, 'w') as f:
        for index, row in corpus_df.iterrows():
            corpus_id = context_to_id[row['context']]
            corpus_entry = {
                "_id": corpus_id,
                "text": row['context'],
                "metadata": {
                    "source": row['source'],
                    "url": row['url1']
                }
            }
            f.write(json.dumps(corpus_entry) + '\n')
    print(f"Successfully created {corpus_filepath} with {len(corpus_df)} entries.")

    # --- 3. Process Queries ---
    # Get unique queries. Per user, queries will already be unique.
    queries_df = positive_df[['query', 'type', 'synthesized']].drop_duplicates(subset=['query']).reset_index(drop=True)

    # Create a mapping from query text to a unique query ID
    query_to_id = {row['query']: f"q{index}" for index, row in queries_df.iterrows()}

    queries_filepath = os.path.join(output_dir, 'queries.jsonl')
    print(f"\nGenerating {queries_filepath}...")
    with open(queries_filepath, 'w') as f:
        for index, row in queries_df.iterrows():
            query_id = query_to_id[row['query']]
            query_entry = {
                "_id": query_id,
                "text": row['query'],
                "metadata": {
                    "type": row['type'],
                    "synthesized": row['synthesized']
                }
            }
            f.write(json.dumps(query_entry) + '\n')
    print(f"Successfully created {queries_filepath} with {len(queries_df)} entries.")


    # --- 4. Create Qrels (Query-Relevance) File ---
    qrels_filepath = os.path.join(qrels_dir, 'test.tsv')
    print(f"\nGenerating {qrels_filepath}...")
    
    # Create a list to hold the relationship data
    qrels_data = []
    # Use the original dataframe to preserve all query-context relationships
    for index, row in positive_df.iterrows():
        query_id = query_to_id.get(row['query'])
        corpus_id = context_to_id.get(row['context'])
        
        if query_id and corpus_id:
            # The format is: query-id, corpus-id, score (assuming 1)
            qrels_data.append([query_id, corpus_id, 1])

    # Create a DataFrame for qrels and save to TSV without duplicates
    qrels_df = pd.DataFrame(qrels_data, columns=['query-id', 'corpus-id', 'score'])
    qrels_df.drop_duplicates(inplace=True)
    qrels_df.to_csv(qrels_filepath, sep='\t', index=False, header=True)
    
    print(f"Successfully created {qrels_filepath} with {len(qrels_df)} relations.")


In [12]:
output_dir = "/rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_subsample_IR_benchmark_v2/"

create_jsonl_with_negative_corpus(query_df, negative_df=negative_corpus_df, output_dir=output_dir)

Generating /rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_subsample_IR_benchmark_v2/corpus.jsonl...
Successfully created /rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_subsample_IR_benchmark_v2/corpus.jsonl with 63885 entries.

Generating /rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_subsample_IR_benchmark_v2/queries.jsonl...
Successfully created /rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_subsample_IR_benchmark_v2/queries.jsonl with 1500 entries.

Generating /rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_subsample_IR_benchmark_v2/qrels/test.tsv...
Successfully created /rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_subsample_IR_benchmark_v2/qrels/test.tsv with 1500 relations.


# testiong the jsonl and qrels files

In [13]:
from datasets import load_dataset

# Load the dataset to verify
corpus = load_dataset(
    "/rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_subsample_IR_benchmark_v2",
    data_files="corpus.jsonl",
    split="train",
)
queries = load_dataset(
    "/rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_subsample_IR_benchmark_v2",
    data_files="queries.jsonl",
    split="train",
)
relevant_docs_data = load_dataset("/rhome/sawale/indus_traning/sentense_transformers/data/stage2_sde/stage2_sde_test_subsample_IR_benchmark_v2", split="test")



Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [14]:
relevant_docs_data.to_pandas()

,query-id,corpus-id,score
0,q0,c0,1
1,q1,c1,1
2,q2,c2,1
3,q3,c3,1
4,q4,c4,1
...,...,...,...
1495,q1495,c1425,1
1496,q1496,c1426,1
1497,q1497,c1427,1
1498,q1498,c1428,1


In [15]:
qdf = queries.to_pandas()
cdf = corpus.to_pandas()


In [16]:
cdf[cdf["_id"] == "c1425"]["text"].values

array(['With actionable Earth observations, the NASA Earth Science Applied Sciences Program empowers communities across the world to find solutions to the challenges they face every day.'],
      dtype=object)

In [17]:
qdf[qdf["_id"] == "q1495"]["text"].values

array(['How does the NASA Earth Science Applied Sciences Program empower communities with actionable Earth observations?'],
      dtype=object)